In [ ]:
# Frames:
#   {B} = world/base frame
#   {S} = FT sensor frame  (site:sensor)
#   {O} = object frame     (site:obj_frame_site / payload_site) 

# 1. AdT needed for backward wrench projection from sensor to object frame.
#  1a. AdT takes in:
#   I.   Object rotation vectors in world/robot/base frame
#   II.  Position of sensor in world frame
#   III. Position of object frame in world frame

rotvecs_B = quat_to_rotvec(quat_B)  # (N,3) object orientations as rotvecs 

p_sensor_B = T_B_sensor[:, :3, 3]   # (N,3) sensor positions in world frame
p_obj_B    = T_B_obj[:, :3, 3]      # (N,3) object positions in world frame

# --- Compute AdT(sensor <- object) for every timestep ---
AdT_sensor_O = get_AdT_sensor_O(rotvecs_B, p_sensor_B, p_obj_B,) # (N,6,6)

# print("AdT_sensor_O shape:", AdT_sensor_O.shape)

# AdT_sensor_O = AdT_sensor_O.transpose(0, 2, 1)  # Invert AdT to get from sensor frame to object frame for backward projection

# 2. Backw-project with AdT to get wrench applied on object in {O} 
#  2a. model_bkwd takes in:
#   I.   Measured wrench in {S} (computed in prev cell)
#   II.  AdT from object frame to sensor frame (computed above)
#   III. Position of Finger (technically point contact on object) in {O}.

p_finger_B = p_B_ee - p_obj_B   # (N,3) position of finger relative to object in world frame
# Transpose the {B} object rotation to correctly orient the {B} finger postion into the object frame
R_O_obj_T = T_B_obj[:, :3, :3].transpose(0, 2, 1)                       # (N,3,3)  
p_finger_O = np.einsum('nij,nj->ni', R_O_obj_T, p_finger_B)             # (N,3)

# --- Back-project: applied wrench on object in {O} ---
w_O_app = model_bkwd_wrench(w_S_meas, AdT_sensor_O, p_finger_O)  # (N,6)

f_O_app   = w_O_app[:, :3]   # (N,3) applied force  in {O}
tau_O_app = w_O_app[:, 3:]   # (N,3) applied torque in {O}

print("Measured wrench {S} at first 5 contact:\n", w_S_meas[:5])
print("Measured wrench {B} at first 5 contact:\n", w_B_meas[:5])
print("Applied wrench {O} at first 5 contact:\n", w_O_app[:5])
print("Finger pos in object frame:\n", p_finger_O[:5])

## Plot the world wrench
fig, ax1, ax2 = plot_wrench_and_tipping(
    t=t_window,
    force_xyz=w_O_app[:, :3],
    torque_primary=w_O_app[:, 4],
    pitch_rad=pitch_B,
    torque_label='tau_y',
    force_labels=('f_x', 'f_y', 'f_z'),
    y_label='Force & Torque (N, Nm)',
    title='Applied wrench in object frame {O}',
)